# 面试题：Agent 如何做可执行的任务分解？

面试回答：把目标拆成带输入、输出、工具、前置条件、风险和验收的原子节点，并以 DAG 表示数据依赖。生成后先做依赖存在与无环校验，运行中逐节点落账。模型只能提议计划，审批、权限、幂等和真实资源状态由确定性控制面负责。下面用采购入职流程验证顺序清单与依赖图的区别。

## 真实案例

为新员工配置电脑涉及预算、选型、下单、审批、通知和资产登记六个业务节点。

## 基线

基线把节点按自然语言顺序执行，忽略下单必须等待预算和审批。

## 结果解读

手写拓扑排序输出每步可执行时刻和依赖。

## 失败案例

循环依赖不能靠模型继续生成，必须在执行前拒绝。

In [1]:
nodes = [{'id':'N1','name':'查预算','deps':[],'write':False}, {'id':'N2','name':'选电脑型号','deps':['N1'],'write':False}, {'id':'N3','name':'创建采购草稿','deps':['N2'],'write':True}, {'id':'N4','name':'经理审批','deps':['N3'],'write':False}, {'id':'N5','name':'提交采购单','deps':['N3','N4'],'write':True}, {'id':'N6','name':'资产登记','deps':['N5'],'write':True}]  # 构造六个带依赖和副作用标记的入职采购节点。
print('节点输入：', [(node['id'], node['name'], node['deps'], node['write']) for node in nodes])  # 输出每个任务的依赖和读写属性。
print('教学说明：采购、审批和资产均为脱敏离线状态，不代表真实企业流程。')  # 明确示例边界。

节点输入： [('N1', '查预算', [], False), ('N2', '选电脑型号', ['N1'], False), ('N3', '创建采购草稿', ['N2'], True), ('N4', '经理审批', ['N3'], False), ('N5', '提交采购单', ['N3', 'N4'], True), ('N6', '资产登记', ['N5'], True)]
教学说明：采购、审批和资产均为脱敏离线状态，不代表真实企业流程。


In [2]:
linear = [node['id'] for node in reversed(nodes)]  # 构造把最后一句先执行的错误线性基线。
violations = [node['id'] for node in nodes if any(dep not in linear[:linear.index(node['id'])] for dep in node['deps'])]  # 检查线性基线中依赖尚未完成的节点。
print('错误线性顺序:', linear, '，依赖违规:', violations)  # 输出盲目顺序执行造成的不可执行节点。

错误线性顺序: ['N6', 'N5', 'N4', 'N3', 'N2', 'N1'] ，依赖违规: ['N2', 'N3', 'N4', 'N5', 'N6']


In [3]:
def topological_plan(graph):  # 定义无需框架的 DAG 静态校验与调度函数。
    remaining = {node['id']:set(node['deps']) for node in graph}  # 复制每个节点尚未满足的依赖集合。
    plan = []  # 初始化按依赖合法排列的执行计划。
    while remaining:  # 在还有未排程节点时持续寻找可执行节点。
        ready = sorted(node_id for node_id, deps in remaining.items() if not deps)  # 找出当前所有依赖为空的节点。
        if not ready:  # 没有可执行节点说明图中存在环或缺失依赖。
            return None, sorted(remaining)  # 返回失败和残留节点供上层拒绝。
        plan.extend(ready)  # 将同一层可并行的节点加入计划。
        for node_id in ready:  # 逐个移除已经排程的节点。
            remaining.pop(node_id)  # 从剩余图删除当前可执行节点。
        for deps in remaining.values():  # 更新未执行节点的依赖集合。
            deps.difference_update(ready)  # 消除已经满足的前置依赖。
    return plan, []  # 返回合法拓扑计划和空错误集。

In [4]:
plan, blocked = topological_plan(nodes)  # 对采购任务图执行静态校验和拓扑排序。
node_map = {node['id']:node['name'] for node in nodes}  # 建立节点标识到业务名称的可读映射。
print('合法计划:', [(node_id, node_map[node_id]) for node_id in plan])  # 输出可执行的业务任务顺序。
print('写操作节点:', [node_id for node_id in plan if next(node['write'] for node in nodes if node['id'] == node_id)])  # 输出需要权限、幂等和审计的副作用节点。
print('同层 N1 之后才会解锁 N2，N3 与 N4 的因果关系由 DAG 而非提示词保证。')  # 解读依赖图带来的控制效果。

合法计划: [('N1', '查预算'), ('N2', '选电脑型号'), ('N3', '创建采购草稿'), ('N4', '经理审批'), ('N5', '提交采购单'), ('N6', '资产登记')]
写操作节点: ['N3', 'N5', 'N6']
同层 N1 之后才会解锁 N2，N3 与 N4 的因果关系由 DAG 而非提示词保证。


In [5]:
cyclic = nodes + [{'id':'N7','name':'预算复核','deps':['N6','N7'],'write':False}]  # 构造自依赖的循环计划反例。
bad_plan, bad_blocked = topological_plan(cyclic)  # 对循环图运行同一静态校验器。
print('失败案例：循环计划=', bad_plan, '，阻塞节点=', bad_blocked)  # 展示执行前识别环而不是运行时无限循环。
print('生产差距：需补充工具 schema、参数来源、权限票据、最大节点/重规划预算和持久化事件账本。')  # 说明教学 DAG 与生产计划器的差距。

失败案例：循环计划= None ，阻塞节点= ['N7']
生产差距：需补充工具 schema、参数来源、权限票据、最大节点/重规划预算和持久化事件账本。


In [6]:
assert plan.index('N5') > plan.index('N4')  # 验证提交采购单不会早于审批。
assert bad_plan is None  # 验证循环任务图会被静态拒绝。
assert len(plan) == 6  # 验证六个业务节点都进入合法计划。